# Decision Tree: Loan Default Prediction

## Problem Statement
Predict whether a loan applicant will default, to support credit-risk decisions.

## Dataset
Applicant/loan-level credit data (`credit.csv`).

## Approach
Preprocessing, then a Decision Tree classifier trained and evaluated on a held-out test split.

## Conclusion
Decision Tree accuracy and structure give an interpretable first baseline for credit-risk classification.


In [131]:
#Build a model that predicts if someone who seeks
# a loan might be a defaulter or a non defaulter.

import pandas as pd
import numpy as np
from sklearn import metrics
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree

In [132]:
data=pd.read_csv('credit.csv')
data.head()

,checking_balance,months_loan_duration,credit_history,purpose,amount,savings_balance,employment_duration,percent_of_income,years_at_residence,age,other_credit,housing,existing_loans_count,job,dependents,phone,default
0,< 0 DM,6,critical,furniture/appliances,1169,unknown,> 7 years,4,4,67,none,own,2,skilled,1,yes,no
1,1 - 200 DM,48,good,furniture/appliances,5951,< 100 DM,1 - 4 years,2,2,22,none,own,1,skilled,1,no,yes
2,unknown,12,critical,education,2096,< 100 DM,4 - 7 years,2,3,49,none,own,1,unskilled,2,no,no
3,< 0 DM,42,good,furniture/appliances,7882,< 100 DM,4 - 7 years,2,4,45,none,other,1,skilled,2,no,no
4,< 0 DM,24,poor,car,4870,< 100 DM,1 - 4 years,3,4,53,none,other,2,skilled,2,no,yes


In [133]:
data.shape

(1000, 17)

In [134]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   checking_balance      1000 non-null   object
 1   months_loan_duration  1000 non-null   int64 
 2   credit_history        1000 non-null   object
 3   purpose               1000 non-null   object
 4   amount                1000 non-null   int64 
 5   savings_balance       1000 non-null   object
 6   employment_duration   1000 non-null   object
 7   percent_of_income     1000 non-null   int64 
 8   years_at_residence    1000 non-null   int64 
 9   age                   1000 non-null   int64 
 10  other_credit          1000 non-null   object
 11  housing               1000 non-null   object
 12  existing_loans_count  1000 non-null   int64 
 13  job                   1000 non-null   object
 14  dependents            1000 non-null   int64 
 15  phone                 1000 non-null   o

In [135]:
# we have too many columns of objects

In [136]:
data.drop('phone',axis=1,inplace=True)

In [137]:
data.head()

,checking_balance,months_loan_duration,credit_history,purpose,amount,savings_balance,employment_duration,percent_of_income,years_at_residence,age,other_credit,housing,existing_loans_count,job,dependents,default
0,< 0 DM,6,critical,furniture/appliances,1169,unknown,> 7 years,4,4,67,none,own,2,skilled,1,no
1,1 - 200 DM,48,good,furniture/appliances,5951,< 100 DM,1 - 4 years,2,2,22,none,own,1,skilled,1,yes
2,unknown,12,critical,education,2096,< 100 DM,4 - 7 years,2,3,49,none,own,1,unskilled,2,no
3,< 0 DM,42,good,furniture/appliances,7882,< 100 DM,4 - 7 years,2,4,45,none,other,1,skilled,2,no
4,< 0 DM,24,poor,car,4870,< 100 DM,1 - 4 years,3,4,53,none,other,2,skilled,2,yes


In [138]:
for i in data.columns:
    if data[i].dtype=='object':
        # when it becomes true we want it to be categorical
        data[i]=pd.Categorical(data[i]) # string items will convert into integer value


In [139]:
# lets take count how many items are there, how many unknown informations are available
print(data.checking_balance.value_counts())


unknown       394
< 0 DM        274
1 - 200 DM    269
> 200 DM       63
Name: checking_balance, dtype: int64


In [140]:
print(data.credit_history.value_counts())

good         530
critical     293
poor          88
very good     49
perfect       40
Name: credit_history, dtype: int64


In [141]:
print(data.purpose.value_counts())

furniture/appliances    473
car                     337
business                 97
education                59
renovations              22
car0                     12
Name: purpose, dtype: int64


In [142]:
print(data.savings_balance.value_counts())

< 100 DM         603
unknown          183
100 - 500 DM     103
500 - 1000 DM     63
> 1000 DM         48
Name: savings_balance, dtype: int64


In [143]:
print(data.housing.value_counts())

own      713
rent     179
other    108
Name: housing, dtype: int64


In [144]:
print(data.job.value_counts())

skilled       630
unskilled     200
management    148
unemployed     22
Name: job, dtype: int64


In [145]:
print(data.purpose.value_counts())

furniture/appliances    473
car                     337
business                 97
education                59
renovations              22
car0                     12
Name: purpose, dtype: int64


In [146]:
replace_data={"checking_balance":{"< 0 DM":1,"1 - 200 DM":2,
                                 "> 200 DM":3,"unknown":-1},
             "credit_history":{"critical":1,"poor":2,
                              "good":3,"very good":4,"perfect":5},
             "savings_balance":{"< 100 DM":1,"100 - 500 DM":2,
                               "500 - 1000 DM":3,"> 1000 DM":4,
                               "unknown":-1},
             "employment_duration":{"unemployed":1,"< 1 year":2,
                                   "1 - 4 years":3,
                                   "4 - 7 years":4,"> 7 years":5},
             "job":{"unemployed":1,"unskilled":2,"skilled":3,
                   "management":4},
            #  "default":{"no":0,"yes":1}
              "purpose":{"car":1,"business":2,"education":3,"renovations":4,
                        "furniture/appliances":5,"car0":6,}


              }


In [147]:
data = data.replace(replace_data)
data.head()

,checking_balance,months_loan_duration,credit_history,purpose,amount,savings_balance,employment_duration,percent_of_income,years_at_residence,age,other_credit,housing,existing_loans_count,job,dependents,default
0,1,6,1,5,1169,-1,5,4,4,67,none,own,2,3,1,no
1,2,48,3,5,5951,1,3,2,2,22,none,own,1,3,1,yes
2,-1,12,1,3,2096,1,4,2,3,49,none,own,1,2,2,no
3,1,42,3,5,7882,1,4,2,4,45,none,other,1,3,2,no
4,1,24,2,1,4870,1,3,3,4,53,none,other,2,3,2,yes


In [148]:
#  remaining categorical colums we need to do on hot encoding
import pandas as pd
on_hot_cols=['housing','other_credit']
data_encoded = pd.get_dummies(data, columns=on_hot_cols)

In [149]:
# now seperate the datas
x=data_encoded.drop('default',axis=1)
y=data_encoded['default']

In [150]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=0)

In [151]:
model = DecisionTreeClassifier(random_state=1)

In [152]:
model.fit(x_train, y_train)

DecisionTreeClassifier(random_state=1)

In [153]:
print(model.score(x_train, y_train))
print(model.score(x_test, y_test))

1.0
0.645


In [154]:
# overfitting model
# decision tree always tends to overfit
# to overcome we use tecnique called as pruning
# Regularization technique used with decision tree---pruning

In [155]:
#pruning in decision trees is used to reduce the size of the tree 
#and improve its generalization ability
model1=DecisionTreeClassifier(criterion='gini',max_depth=3,random_state=1)

In [156]:
model1.fit(x_train,y_train)

DecisionTreeClassifier(max_depth=3, random_state=1)

In [157]:
print("Accuracy on training set:",model1.score(x_train,y_train))
print("Accuracy on test set:",model1.score(x_test,y_test))

Accuracy on training set: 0.735
Accuracy on test set: 0.685


In [158]:
y_predict=model1.predict(x_test)

In [159]:
confusion_matrix_data = metrics.confusion_matrix(y_test, y_predict)

In [160]:
d = pd.DataFrame(confusion_matrix_data, index= [i for i in ['Yes', 'No']], columns = [i for i in ['Yes', 'No']])

In [161]:
print(d)

     Yes  No
Yes  133   9
No    54   4


In [162]:
from sklearn.ensemble import RandomForestClassifier

In [163]:
model2=RandomForestClassifier(n_estimators=12,random_state=1,max_features=10)

In [164]:
model2.fit(x_train,y_train)

RandomForestClassifier(max_features=10, n_estimators=12, random_state=1)

In [165]:
print("Random Forest Classifier Score:",model2.score(x_test,y_test))

Random Forest Classifier Score: 0.75


In [166]:
result=pd.DataFrame(model1.feature_importances_,columns=["Imp"],index=x_train.columns)

In [167]:
result

,Imp
checking_balance,0.444118
months_loan_duration,0.152877
credit_history,0.207304
purpose,0.050946
amount,0.000000
savings_balance,0.000000
employment_duration,0.000000
percent_of_income,0.000000
years_at_residence,0.000000
age,0.026681


In [168]:
from sklearn.ensemble import AdaBoostClassifier

In [169]:
ada_boost_model=AdaBoostClassifier(n_estimators=10,random_state=1)

In [170]:
ada_boost_model.fit(x_train,y_train)

AdaBoostClassifier(n_estimators=10, random_state=1)

In [171]:
ada_boost_model.score(x_train,y_train)

0.77125

In [172]:
y_predict = ada_boost_model.predict(x_test)

In [173]:
cm = metrics.confusion_matrix(y_test, y_predict)

In [174]:
d = pd.DataFrame(cm, index=[i for i in['yes','no']],
                 columns = [i for i in ['yes','no']])
d

,yes,no
yes,120,22
no,34,24


In [175]:
confusion_matrix_data

array([[133,   9],
       [ 54,   4]], dtype=int64)

In [176]:
tp=133
fp=54
fn=9
tn=4

In [177]:
print('Accuracy =',(tp+tn)/(tp+tn+fp+fn))

Accuracy = 0.685


In [178]:
print('Recall')

Recall


In [179]:
from sklearn.ensemble import GradientBoostingClassifier

In [180]:
gradient_model=GradientBoostingClassifier(n_estimators=50,random_state=1)

In [181]:
gradient_model.fit(x_train,y_train)

GradientBoostingClassifier(n_estimators=50, random_state=1)

In [182]:
y_pred=gradient_model.predict(x_test)

In [183]:
cm=metrics.confusion_matrix(y_test, y_predict)
d = pd.DataFrame(cm, index=[i for i in['yes','no']],
                 columns = [i for i in ['yes','no']])

In [184]:
d

,yes,no
yes,120,22
no,34,24


In [185]:
confusion_matrix_data

array([[133,   9],
       [ 54,   4]], dtype=int64)

In [186]:
tp=133
fp=54
fn=9
tn=4

In [187]:
print('Accuracy =',(tp+tn)/(tp+tn+fp+fn))

Accuracy = 0.685


In [188]:
#Decision tree----75
#Random forest----76
#Ada Boost----74
#gradient Boost----